# Week 47, Day 3: Communication through coherence

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w047/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Gain against phase relation

Write gain(dphi): draw 2,000 sender spikes at times k/f minus dphi/(2 pi f) plus 2 ms jitter, add the 4 ms delay, and return the mean receiver excitability at arrival. Evaluate it for 24 phase relations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2015)
f, T, delay = 40.0, 2.0, 0.004
excitability = lambda t: 0.5 * (1 + np.cos(2 * np.pi * f * t))

def gain(dphi, jitter=0.002, n=2000):
    k = r.integers(0, int(T * f), n)
    t_spike = ...                              # sender fires at its phase 0, leading the receiver by dphi
    return ...

phis = np.linspace(0, 2 * np.pi, 24, endpoint=False)
gains = np.array([gain(p) for p in phis])
print(f"gain ranges from {gains.min():.2f} to {gains.max():.2f}; best phase relation {phis[np.argmax(gains)]:.2f} rad (2 pi f delay = {2 * np.pi * f * delay:.2f})")
plt.plot(phis, gains, "o-"); plt.xlabel("sender lead (rad)"); plt.ylabel("effective gain"); plt.show()

In [ ]:
import numpy as np
def test_range():
    "gain varies by more than a factor of 4 across phase relations"
    assert gains.max() > 4 * gains.min()
def test_optimum():
    "the best phase relation compensates the delay: within 0.4 rad of 2 pi f delay"
    best = phis[np.argmax(gains)]
    assert abs(np.angle(np.exp(1j * (best - 2 * np.pi * 40 * 0.004)))) < 0.4

test_range()
test_optimum()
print("All checks passed")

### Exercise: Coherent wins, incoherent averages out

Compute gA with the best phase relation from Step 2; for sender B draw 2,000 spikes at times k / 37 Hz with 2 ms jitter and the same delay. Then recompute A's gain with jitter of 5 and 10 ms.

In [ ]:
best = phis[np.argmax(gains)]
gA = gain(best)
kB = r.integers(0, int(T * 37), 2000)
gB = ...
jit = {j: gain(best, jitter=j) for j in [0.002, 0.005, 0.010]}
print(f"coherent sender gain {gA:.2f}; incoherent sender gain {gB:.2f}; ratio {gA / gB:.2f}")
print("coherent sender with more jitter:", ", ".join(f"{1000 * j:.0f} ms: {v:.2f}" for j, v in jit.items()))

In [ ]:
import numpy as np
def test_route():
    "the coherent sender's gain is more than 1.6 times the incoherent sender's"
    assert gA > 1.6 * gB
def test_avg():
    "the incoherent sender's gain is close to the average excitability, 0.5"
    assert abs(gB - 0.5) < 0.05
def test_jitter():
    "with 10 ms jitter, the coherent advantage disappears (ratio below 1.15)"
    assert jit[0.010] / gB < 1.15

test_route()
test_avg()
test_jitter()
print("All checks passed")